# QRT-style ML mock 1: cement-strength classification

**Core time:** 45-60 minutes

**Optional extension:** 20-30 minutes
**Goal:** build and evaluate a leakage-safe binary classifier using scikit-learn.

This is an original practice exercise based on your recollection and public concrete-strength variables. It is not an official or leaked QRT question.

## Scenario and rules

You receive `train_data` and `test_data`. The final column, `is_strong`, is `1.0` for strong material and `0.0` for weak material.

- Use `sample_id` and `production_date` only as metadata, never as model inputs.
- Use a stratified train/validation split for model selection.
- Fit imputation and scaling **inside** a scikit-learn `Pipeline`.
- Do not inspect test labels or test performance until the final model is fixed.
- Compare against a simple baseline and report more than accuracy.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 20261101
FEATURES = ["cement_kg_m3", "water_kg_m3", "blast_furnace_slag_kg_m3", "superplasticizer_kg_m3", "curing_days"]
TARGET = "is_strong"
ROOT = Path.cwd()
if not (ROOT / "data" / "cement_train.csv").exists():
    ROOT = ROOT / "QRT_mock_test"
train_data = pd.read_csv(ROOT / "data" / "cement_train.csv", parse_dates=["production_date"])
test_data = pd.read_csv(ROOT / "data" / "cement_test.csv", parse_dates=["production_date"])
print("train shape:", train_data.shape)
print("protected test shape:", test_data.shape)
display(train_data.head())

def report_check(name, function):
    try:
        function()
        print(f"PASS: {name}")
    except NotImplementedError:
        print(f"NOT YET: {name}")
    except Exception as exc:
        print(f"CHECK: {name} -> {type(exc).__name__}: {exc}")

## Task 1 - Audit the training data

Implement `audit_classification_data`. Return a dictionary containing:

- `train_shape` and `test_shape`;
- `duplicate_train_ids`;
- `missing_train` as a per-column Series;
- `train_class_balance` as normalized target frequencies;
- `feature_columns` containing exactly the five permitted model features.

You may inspect the test schema and missing counts, but do not inspect its target distribution.

In [ ]:
def audit_classification_data(train_data: pd.DataFrame, test_data: pd.DataFrame) -> dict[str, object]:
    # TODO: audit shapes, IDs, missingness, training balance, and feature selection.
    raise NotImplementedError

audit = None
try:
    audit = audit_classification_data(train_data, test_data)
    display(pd.Series({"train_rows": audit["train_shape"][0], "test_rows": audit["test_shape"][0], "duplicate_train_ids": audit["duplicate_train_ids"]}))
    display(audit["missing_train"].to_frame("missing"))
    display(audit["train_class_balance"].to_frame("share"))
except NotImplementedError:
    print("Complete audit_classification_data, then rerun this cell.")

In [ ]:
def check_audit():
    result = audit_classification_data(train_data, test_data)
    assert result["train_shape"] == (1200, 8)
    assert result["test_shape"] == (300, 8)
    assert result["duplicate_train_ids"] == 0
    assert list(result["feature_columns"]) == FEATURES
    assert set(result["train_class_balance"].index.astype(float)) == {0.0, 1.0}
    assert result["missing_train"].sum() > 0

report_check("classification data audit", check_audit)

## Task 2 - Build the stratified split and pipeline

Implement:

1. `make_classification_split` using a 25% validation fraction, stratification and `RANDOM_STATE`.
2. `build_classifier` returning a pipeline with median imputation, standardization and logistic regression.

The scaler must never be fitted before the split.

In [ ]:
def make_classification_split(train_data: pd.DataFrame):
    # TODO: return X_train, X_validation, y_train, y_validation.
    raise NotImplementedError

def build_classifier() -> Pipeline:
    # TODO: return SimpleImputer -> StandardScaler -> LogisticRegression.
    raise NotImplementedError

In [ ]:
def check_split_and_pipeline():
    X_train, X_validation, y_train, y_validation = make_classification_split(train_data)
    assert list(X_train.columns) == FEATURES
    assert len(X_train) == 900 and len(X_validation) == 300
    assert abs(y_train.mean() - y_validation.mean()) < 0.01
    model = build_classifier()
    assert isinstance(model, Pipeline)
    assert [name for name, _ in model.steps] == ["imputer", "scaler", "model"]
    model.fit(X_train, y_train)
    assert set(model.predict(X_validation)).issubset({0, 1})

report_check("stratified split and pipeline", check_split_and_pipeline)

## Task 3 - Compare the baseline and classifier

Implement `classification_metrics` with these keys:

`accuracy`, `balanced_accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `confusion_matrix`.

Fit a prior-frequency `DummyClassifier` and your logistic pipeline on the training partition. Compare them on validation data and select the model using validation results only.

In [ ]:
def classification_metrics(y_true, y_pred, y_probability) -> dict[str, object]:
    # TODO: calculate the seven requested outputs.
    raise NotImplementedError

validation_results = None
classifier = None
try:
    X_train, X_validation, y_train, y_validation = make_classification_split(train_data)
    dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
    classifier = build_classifier().fit(X_train, y_train)
    validation_results = pd.DataFrame(
        {
            "dummy": classification_metrics(y_validation, dummy.predict(X_validation), dummy.predict_proba(X_validation)[:, 1]),
            "logistic": classification_metrics(y_validation, classifier.predict(X_validation), classifier.predict_proba(X_validation)[:, 1]),
        }
    ).T
    display(validation_results.drop(columns="confusion_matrix"))
    print("Logistic confusion matrix:\n", validation_results.loc["logistic", "confusion_matrix"])
except NotImplementedError:
    print("Complete Tasks 2-3, then rerun this cell.")

In [ ]:
def check_validation_metrics():
    X_train, X_validation, y_train, y_validation = make_classification_split(train_data)
    model = build_classifier().fit(X_train, y_train)
    metrics = classification_metrics(y_validation, model.predict(X_validation), model.predict_proba(X_validation)[:, 1])
    assert set(metrics) == {"accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "confusion_matrix"}
    assert np.asarray(metrics["confusion_matrix"]).shape == (2, 2)
    assert metrics["roc_auc"] > 0.85
    assert metrics["f1"] > 0.75

report_check("classification metrics", check_validation_metrics)

## Task 4 - Final protected-test evaluation and model extraction

After fixing the model, refit it on all training rows and inspect the test labels exactly once.

Implement:

- `make_test_predictions`, returning `sample_id`, `production_date`, `actual`, `predicted`, and `probability_strong`;
- `coefficient_table`, returning each feature's standardized coefficient and absolute coefficient, sorted by importance.

In [ ]:
def make_test_predictions(model: Pipeline, test_data: pd.DataFrame, threshold: float = 0.5) -> pd.DataFrame:
    # TODO: build the requested prediction audit table.
    raise NotImplementedError

def coefficient_table(model: Pipeline) -> pd.DataFrame:
    # TODO: extract standardized logistic coefficients from the fitted pipeline.
    raise NotImplementedError

test_predictions = None
test_metrics = None
try:
    final_model = build_classifier().fit(train_data[FEATURES], train_data[TARGET].astype(int))
    test_predictions = make_test_predictions(final_model, test_data)
    test_metrics = classification_metrics(test_predictions["actual"], test_predictions["predicted"], test_predictions["probability_strong"])
    display(pd.Series({key: value for key, value in test_metrics.items() if key != "confusion_matrix"}).to_frame("test"))
    print("Test confusion matrix:\n", test_metrics["confusion_matrix"])
    display(coefficient_table(final_model))
    display(test_predictions.head())
except NotImplementedError:
    print("Complete Task 4 only after model selection is finished.")

In [ ]:
def check_final_classification_outputs():
    model = build_classifier().fit(train_data[FEATURES], train_data[TARGET].astype(int))
    predictions = make_test_predictions(model, test_data)
    coefficients = coefficient_table(model)
    assert list(predictions.columns) == ["sample_id", "production_date", "actual", "predicted", "probability_strong"]
    assert len(predictions) == len(test_data)
    assert predictions["probability_strong"].between(0.0, 1.0).all()
    assert set(coefficients["feature"]) == set(FEATURES)

report_check("final classification outputs", check_final_classification_outputs)

## Optional advanced section

These cells are not required for the core checks.

1. Choose a probability threshold using **validation F1 only**, then apply it once to the test probabilities.
2. Compare logistic regression with a median-imputed `RandomForestClassifier` using validation ROC-AUC and F1.
3. Explain why threshold tuning on test data would be leakage and why accuracy alone can be misleading.

In [ ]:
# OPTIONAL TODO: threshold search and RandomForest comparison.
advanced_classification_results = None
print("Optional section not run." if advanced_classification_results is None else advanced_classification_results)

## Final interpretation

In three or four sentences, state:

- whether the logistic model beat the dummy baseline;
- which error type matters more if accepting weak cement is costly;
- which features had the largest standardized effects;
- one reason live performance could be worse than this synthetic test.

**Your answer:** TODO.